# Hard Thresholding coarse search on Fritz

This notebook uses the matrix-free implementation from `hard_thresholding_hpc.py`. The completed user-movie matrix is never materialized.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.sparse.linalg import ArpackNoConvergence

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hard_thresholding_hpc import (
    hard_thresholding_model_matrix_free,
    predict_pairs,
)

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml-latest"
RESULTS_DIR = PROJECT_ROOT / "results" / "hard_thresholding_coarse"
HISTORIES_DIR = RESULTS_DIR / "histories"
HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_ratings.csv")
user_map = pd.read_csv(PROCESSED_DIR / "user_map.csv")
movie_map = pd.read_csv(PROCESSED_DIR / "movie_map.csv")

n_users = len(user_map)
n_movies = len(movie_map)

train_fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

val_user_idx = val_df["user_idx"].to_numpy(dtype=np.int64)
val_movie_idx = val_df["movie_idx"].to_numpy(dtype=np.int64)
val_ratings = val_df["rating"].to_numpy(dtype=np.float64)

print("Train fit shape:", train_fit_df.shape)
print("Validation shape:", val_df.shape)
print("Number of users:", n_users)
print("Number of movies:", n_movies)

In [ ]:
def hard_validation_metrics(U, singular_values, V):
    predictions = predict_pairs(
        U,
        singular_values,
        V,
        val_user_idx,
        val_movie_idx,
    )
    predictions = np.clip(predictions, 0.5, 5.0)
    return {
        "val_rmse": float(np.sqrt(np.mean((val_ratings - predictions) ** 2))),
    }

## Coarse rank grid

Each rank is trained once. Validation RMSE at every iteration selects `n_iter` without rerunning the same rank.

In [ ]:
hard_rank_values = [4, 8, 16, 32, 64]
hard_grid_max_iter = 15
hard_svd_tol = 1e-4
hard_svd_maxiter = 300

# Compile the prediction kernel before the timed search.
predict_pairs(
    np.ones((2, 1)),
    np.ones(1),
    np.ones((2, 1)),
    np.array([0], dtype=np.int64),
    np.array([0], dtype=np.int64),
)

hard_histories = []
hard_failed_configs = []
hard_search_start = perf_counter()

for config_number, rank in enumerate(hard_rank_values, start=1):
    history_path = HISTORIES_DIR / f"rank-{rank}.csv"
    print(f"[{config_number}/{len(hard_rank_values)}] rank={rank}")

    if history_path.exists():
        print("  using saved history")
        hard_histories.append(pd.read_csv(history_path))
        continue

    config_start = perf_counter()
    try:
        _, _, _, history_df = hard_thresholding_model_matrix_free(
            train_fit_df,
            n_users=n_users,
            n_movies=n_movies,
            rank=rank,
            n_iter=hard_grid_max_iter,
            metrics_fn=hard_validation_metrics,
            svd_tol=hard_svd_tol,
            svd_maxiter=hard_svd_maxiter,
            random_state=42,
        )
    except ArpackNoConvergence as error:
        hard_failed_configs.append({
            "rank": rank,
            "error": repr(error),
        })
        print("  failed:", repr(error))
        continue

    history_df["config_seconds"] = perf_counter() - config_start
    history_df.to_csv(history_path, index=False)
    hard_histories.append(history_df)

if not hard_histories:
    raise RuntimeError("Every hard-thresholding configuration failed")

hard_tuning_results_df = pd.concat(hard_histories, ignore_index=True)
hard_best_by_rank = (
    hard_tuning_results_df.loc[
        hard_tuning_results_df.groupby("rank")["val_rmse"].idxmin()
    ]
    .sort_values("val_rmse")
    .reset_index(drop=True)
)
hard_tuning_results_df.to_csv(RESULTS_DIR / "all_histories.csv", index=False)
hard_best_by_rank.to_csv(RESULTS_DIR / "best_by_rank.csv", index=False)

if hard_failed_configs:
    pd.DataFrame(hard_failed_configs).to_csv(
        RESULTS_DIR / "failed_configs.csv",
        index=False,
    )

print(f"Coarse search time: {(perf_counter() - hard_search_start) / 60:.1f} min")
print("Failed configurations:", len(hard_failed_configs))

## Coarse-search summary

In [ ]:
print("Coarse search completed.")
display(hard_best_by_rank)